# KPI & Retention Analysis
Reads reproducible pipeline outputs. Business logic remains in `src/analytics`; this notebook is the interpretation layer.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
monthly = pd.read_csv(ROOT/'data/processed/monthly_kpis.csv', parse_dates=['month'])
cohort = pd.read_csv(ROOT/'data/processed/cohort_retention.csv', index_col='cohort')
by_channel = pd.read_csv(ROOT/'data/processed/retention_by_channel.csv')
by_device = pd.read_csv(ROOT/'data/processed/retention_by_device.csv')
monthly.tail()

## KPI trends

In [ ]:
metric_candidates = [c for c in ['mau','active_users','orders','net_revenue','arpu'] if c in monthly.columns]
fig, axes = plt.subplots(len(metric_candidates), 1, figsize=(12, 3*len(metric_candidates)), sharex=True)
axes = [axes] if len(metric_candidates) == 1 else axes
for metric, ax in zip(metric_candidates, axes):
    monthly.plot(x='month', y=metric, marker='o', ax=ax, legend=False, title=metric.replace('_',' ').title())
    ax.grid(alpha=.2)
plt.tight_layout()

## Cohort retention heatmap
Blank cells are right-censored periods that have not yet had time to mature; they should not be filled with zero.

In [ ]:
plt.figure(figsize=(14,7))
sns.heatmap(cohort, cmap='Blues', vmin=0, vmax=1, mask=cohort.isna(), cbar_kws={'label':'Retention'})
plt.title('Cohort retention by months since signup')
plt.xlabel('Months since signup'); plt.ylabel('Signup cohort')
plt.tight_layout()

## Segment comparisons
Channel and device cuts help separate aggregate retention from acquisition-mix effects. They remain descriptive because the synthetic generator is not a randomized experiment.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4))
by_channel.set_index('channel').plot.bar(ax=axes[0], title='Retention by acquisition channel')
by_device.set_index('device').plot.bar(ax=axes[1], title='Retention by device')
for ax in axes: ax.set_ylabel('Retention'); ax.set_ylim(0,1); ax.grid(axis='y', alpha=.2)
plt.tight_layout()

## Interpretation checklist
- Compare mature cohorts at the same age, never diagonal calendar positions.
- Treat recent blank cells as right-censored, not churn.
- Use segment gaps to form hypotheses, then validate them with real data or experiments.